<a href="https://colab.research.google.com/github/busycaesar/Attention_Mechanism/blob/Master/QKV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers

In [ ]:
from transformers import GPT2Tokenizer, GPT2Model

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2Model.from_pretrained('gpt2', output_attentions=True)
model.eval()

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [ ]:
print('GPT-2 loaded.')
print(f'Layers : {model.config.n_layer}')
print(f'Heads  : {model.config.n_head}')
print(f'd_model: {model.config.n_embd}')
print(f'd_k    : {model.config.n_embd // model.config.n_head}  (d_model / n_heads)')

GPT-2 loaded.
Layers : 12
Heads  : 12
d_model: 768
d_k    : 64  (d_model / n_heads)


In [ ]:
sentence = 'The dog did not cross the road because it was tired'
inputs   = tokenizer(sentence, return_tensors='pt')
input_ids = inputs['input_ids']

tokens = [tokenizer.decode([tid]) for tid in input_ids[0]]
# print('Tokens:', tokens)
# print('IDs   :', input_ids[0].tolist())

it_idx   = tokens.index(' it')
dog_idx  = tokens.index(' dog')
road_idx = tokens.index(' road')
# print(f'\n"it"   is token index {it_idx}')
# print(f'"dog"  is token index {dog_idx}')
# print(f'"road" is token index {road_idx}')

In [ ]:
!pip install torch -q

In [ ]:
import torch

# Step 4 — Hook into GPT-2 to extract Q, K, V tensors

qkv_store = {}

def make_hook(layer_idx):
    def hook(module, input, output):
        with torch.no_grad():
            x   = input[0]
            qkv = module.c_attn(x)
            d   = model.config.n_embd
            Q, K, V = qkv.split(d, dim=2)
            qkv_store[layer_idx] = (
                Q.squeeze(0).detach(),
                K.squeeze(0).detach(),
                V.squeeze(0).detach()
            )
    return hook

# --- FIX: handle both old and new HuggingFace versions ---
hooks = []
try:
    blocks = model.transformer.h        # older versions
except AttributeError:
    blocks = model.h                    # newer versions

for i, block in enumerate(blocks):
    h = block.attn.register_forward_hook(make_hook(i))
    hooks.append(h)

with torch.no_grad():
    outputs = model(**inputs)

for h in hooks:
    h.remove()

print(f'Q, K, V captured for {len(qkv_store)} layers.')
print(f'Shape of Q at layer 0: {qkv_store[0][0].shape}')

Q, K, V captured for 12 layers.
Shape of Q at layer 0: torch.Size([11, 768])


In [ ]:
# Step 5 — Print Q, K, V vectors for every token (Layer 0, first 8 dims for readability)
# d_model = 768, printing all 768 would be unreadable.
# We show the first 8 dimensions as a window into the full vector.

LAYER = 0
Q, K, V = qkv_store[LAYER]

print(f'=== Layer {LAYER} — Q, K, V for each token (first 8 of 768 dims) ===\n')
for i, tok in enumerate(tokens):
    q_preview = Q[i, :8].numpy().round(3)
    k_preview = K[i, :8].numpy().round(3)
    v_preview = V[i, :8].numpy().round(3)
    print(f'Token {i:2d}  "{tok}"')
    print(f'  Q: {q_preview}')
    print(f'  K: {k_preview}')
    print(f'  V: {v_preview}')
    print()

=== Layer 0 — Q, K, V for each token (first 8 of 768 dims) ===

Token  0  "The"
  Q: [ 0.153 -0.415  0.367 -1.178  0.168  0.663 -0.685  0.019]
  K: [-0.942  1.902  0.872 -0.15   0.404  0.208 -0.117  0.623]
  V: [-0.013 -0.014  0.127 -0.022 -0.208  0.098  0.129 -0.087]

Token  1  " dog"
  Q: [-0.147  1.144  0.227  0.275 -0.33  -0.293  0.935  0.974]
  K: [-1.43   3.745  1.287 -0.942  1.399  0.659 -0.658  0.119]
  V: [-0.074 -0.004 -0.081  0.255 -0.094 -0.084 -0.077 -0.194]

Token  2  " did"
  Q: [ 0.056 -0.437 -0.303 -0.292 -0.049 -1.079 -1.458  0.353]
  K: [-1.867  1.798  2.238 -0.09   1.294 -0.309  1.495 -0.452]
  V: [ 0.175 -0.134  0.039 -0.004  0.142  0.365  0.535 -0.137]

Token  3  " not"
  Q: [-0.291 -0.803  0.108 -0.265 -0.493  0.084 -0.593  0.232]
  K: [-2.327  2.4    1.713  0.339  1.304  0.16   0.968  0.799]
  V: [-0.029 -0.025 -0.03   0.02   0.005  0.533 -0.025  0.139]

Token  4  " cross"
  Q: [ 0.556  0.412 -1.032 -0.782 -0.055 -0.236  1.263 -0.189]
  K: [-2.738  2.512  2.6   

In [ ]:
# # Step 6 — Manual dot product PER HEAD (correct approach)
# # GPT-2: d_model=768, n_heads=12, d_k=64 per head
# # Q shape: (seq_len, 768) — split into 12 heads of 64 dims each

# import math

# n_heads = model.config.n_head        # 12
# d_k     = model.config.n_embd // n_heads  # 64
# Q, K, V = qkv_store[0]

# print(f'd_k per head = {d_k}, sqrt(d_k) = {math.sqrt(d_k):.3f}\n')

# # Split Q and K into heads
# # Each head gets columns [head*d_k : (head+1)*d_k]
# results = {}  # head -> attention weights

# for head in range(n_heads):
#     start = head * d_k
#     end   = start + d_k

#     q_it_head = Q[it_idx, start:end]   # (64,)

#     raw_scores = []
#     for i in range(len(tokens)):
#         k_j = K[i, start:end]          # (64,)
#         raw = torch.dot(q_it_head, k_j).item()
#         raw_scores.append(raw)

#     scaled  = [s / math.sqrt(d_k) for s in raw_scores]
#     weights = torch.softmax(torch.tensor(scaled), dim=0).numpy()
#     results[head] = weights

#     dog_w  = weights[dog_idx]
#     road_w = weights[road_idx]
#     winner = 'dog > road ✓' if dog_w > road_w else 'road > dog'
#     print(f'Head {head:2d}:  dog={dog_w:.4f}  road={road_w:.4f}  → {winner}')

# # Find the head where dog wins most clearly
# best_head = max(range(n_heads),
#     key=lambda h: results[h][dog_idx] - results[h][road_idx])
# print(f'\nHead {best_head} shows the clearest dog > road separation.')
# print(f'\n=== Full attention weights for "it" — Head {best_head} ===\n')
# for i, tok in enumerate(tokens):
#     w      = results[best_head][i]
#     bar    = '█' * int(w * 300)
#     marker = ' <-- dog' if i == dog_idx else (' <-- road' if i == road_idx else (' <-- it' if i == it_idx else ''))
#     print(f'  "{tok:10s}"  {w:.4f}  {bar}{marker}')

In [ ]:
# # Step 7 — Compare with GPT-2's own internal attention weights
# # outputs.attentions shape: (n_layers, 1, n_heads, seq_len, seq_len)
# # Row i, col j = how much token i attends to token j

# LAYER = 0
# HEAD  = 0  # try heads 0-11

# attn = outputs.attentions[LAYER][0][HEAD].detach().numpy()  # (seq_len, seq_len)

# print(f'=== GPT-2 internal attention weights (Layer {LAYER}, Head {HEAD}) ===\n')
# print(f'Row = query token, Col = key token')
# print(f'Reading row for "it" (index {it_idx}):\n')

# it_row = attn[it_idx]
# for i, tok in enumerate(tokens):
#     bar    = '█' * int(it_row[i] * 200)
#     marker = '  <-- "it"' if i == it_idx else ('  <-- "dog"' if i == dog_idx else ('  <-- "road"' if i == road_idx else ''))
#     print(f'  "{tok:10s}"  {it_row[i]:.4f}  {bar}{marker}')

# print(f'\n  Internal weight on "dog" : {it_row[dog_idx]:.4f}')
# print(f'  Internal weight on "road": {it_row[road_idx]:.4f}')
# print(f'\n  Note: GPT-2 uses causal (masked) attention — tokens can only attend')
# print(f'  to previous tokens. "it" is at index {it_idx}, so it cannot see tokens after it.')

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# # Step 8 — Heatmap: attention weights across all heads for "it"
# # Shows how differently each head distributes attention from "it"

# LAYER = 6
# n_heads = model.config.n_head  # 12
# n_tokens = len(tokens)

# # Build matrix: rows = heads, cols = tokens
# heatmap = np.zeros((n_heads, n_tokens))
# for head in range(n_heads):
#     heatmap[head] = outputs.attentions[LAYER][0][head][it_idx].detach().numpy()

# fig, ax = plt.subplots(figsize=(12, 5))
# im = ax.imshow(heatmap, aspect='auto', cmap='YlOrRd')

# ax.set_xticks(range(n_tokens))
# ax.set_xticklabels([f'"{t.strip()}"' for t in tokens], rotation=30, ha='right', fontsize=10)
# ax.set_yticks(range(n_heads))
# ax.set_yticklabels([f'Head {i}' for i in range(n_heads)], fontsize=9)
# ax.set_title(f'Layer {LAYER} — Attention weights FROM "it" TO every token, across all 12 heads', fontsize=12, pad=12)
# ax.set_xlabel('Key token (what "it" attends to)', fontsize=10)
# ax.set_ylabel('Attention head', fontsize=10)

# # Annotate dog and road columns
# for col, label, color in [(dog_idx, 'dog', 'green'), (road_idx, 'road', 'red')]:
#     ax.axvline(col, color=color, linewidth=1.5, linestyle='--', alpha=0.6)
#     ax.text(col, -0.8, f'↑ {label}', color=color, ha='center', fontsize=9, fontweight='bold')

# plt.colorbar(im, ax=ax, label='Attention weight')
# plt.tight_layout()
# plt.savefig('attention_heatmap.png', dpi=150, bbox_inches='tight')
# plt.show()
# print('Heatmap saved as attention_heatmap.png')

In [ ]:
# # Step 9 — Geometric verification
# # Confirm: dot_product = |q| * |k| * cos(theta)

# print('=== Geometric verification: dot product = |q||k|cos(theta) ===\n')

# for label, idx in [('dog', dog_idx), ('road', road_idx)]:
#     q  = Q[it_idx].float()
#     k  = K[idx].float()
#     dot      = torch.dot(q, k).item()
#     mag_q    = torch.norm(q).item()
#     mag_k    = torch.norm(k).item()
#     cos_t    = dot / (mag_q * mag_k)
#     theta    = math.degrees(math.acos(max(-1.0, min(1.0, cos_t))))
#     verified = mag_q * mag_k * cos_t

#     print(f'  "it" -> "{label}"')
#     print(f'    |q_it|           = {mag_q:.4f}')
#     print(f'    |k_{label}|      = {mag_k:.4f}')
#     print(f'    cos(theta)       = {cos_t:.4f}')
#     print(f'    theta            = {theta:.2f} degrees')
#     print(f'    |q||k|cos(theta) = {verified:.4f}')
#     print(f'    dot product      = {dot:.4f}  (matches: {abs(verified - dot) < 0.01})')
#     print()

# print('  Smaller angle = more aligned = higher attention score.')
# print(f'  theta(it->dog) vs theta(it->road) — which is smaller? Check above.')

In [ ]:
# # Full attention distribution from "it" — every token, no cherry-picking
# import math, torch

# L, H = 4, 3
# d_head = model.config.n_embd // model.config.n_head

# Q, K, V = qkv_store[L]
# q_it = Q[it_idx, H*d_head:(H+1)*d_head]

# print(f'Layer {L}, Head {H} — "it" scored against every token\n')
# print(f'{"idx":<5}{"token":<12}{"q·k":>10}{"scaled":>10}{"softmax":>10}   bar')
# print('-' * 62)

# raw = []
# for i in range(len(tokens)):
#     k_i = K[i, H*d_head:(H+1)*d_head]
#     raw.append(torch.dot(q_it, k_i).item())

# scaled = [r / math.sqrt(d_head) for r in raw]

# # causal mask — "it" cannot see tokens after it
# masked = [s if i <= it_idx else float('-inf') for i, s in enumerate(scaled)]
# weights = torch.softmax(torch.tensor(masked), dim=0)

# for i, tok in enumerate(tokens):
#     if i > it_idx:
#         print(f'{i:<5}{tok.strip():<12}{"":>10}{"masked":>10}{"—":>10}')
#         continue
#     bar = '█' * int(weights[i].item() * 60)
#     print(f'{i:<5}{tok.strip():<12}{raw[i]:>10.2f}{scaled[i]:>10.3f}{weights[i]:>10.4f}   {bar}')

# print(f'\nsum of weights = {weights.sum():.4f}')
# print(f'\nRanked:')
# order = sorted(range(it_idx+1), key=lambda i: -weights[i])
# for rank, i in enumerate(order, 1):
#     print(f'  {rank}. "{tokens[i].strip()}"  {weights[i]:.4f}')

In [ ]:
import torch, math

n_layers = model.config.n_layer
n_heads  = model.config.n_head
prior    = list(range(it_idx + 1))
names    = [tokens[i].strip() for i in prior]

hits = []

for L in range(n_layers):
    print(f'\nLAYER {L}')
    hdr = (f'{"Hd":<4}' + ''.join(f'{n:>9}' for n in names)
           + f'{"pick":>10}{"wt":>7}{"margin":>8}{"entropy":>9}  verdict')
    print(hdr); print('-'*len(hdr))

    for H in range(n_heads):
        w = outputs.attentions[L][0][H][it_idx][:len(prior)]

        ns = w.clone(); ns[0] = -1                 # drop position-0 sink
        i1 = ns.argmax().item(); w1 = w[i1].item()
        ns2 = ns.clone(); ns2[i1] = -1
        w2 = ns2.max().item()                      # runner-up
        margin = w1 - w2

        p = w[w > 0]
        ent = float(-(p * p.log2()).sum())         # bits; low = focused

        if   margin > 0.30: v = 'decisive'
        elif margin > 0.10: v = 'clear'
        elif margin > 0.03: v = 'weak'
        else:               v = 'no signal'

        cells = ''.join(f'{x:>9.3f}' for x in w.tolist())
        flag = '  <<<' if names[i1]=='dog' and w[dog_idx]>w[road_idx] and margin>0.10 else ''
        print(f'{H:<4}{cells}{names[i1]:>10}{w1:>7.3f}{margin:>8.3f}{ent:>9.2f}  {v}{flag}')

        if names[i1]=='dog' and w[dog_idx]>w[road_idx]:
            hits.append((margin, L, H, w[dog_idx].item(), w[road_idx].item(), ent))

hits.sort(reverse=True)
print('\nHeads picking "dog", ranked by margin:\n')
for m,L,H,d,r,e in hits:
    print(f'  L{L:<2} H{H:<2}  dog={d:.3f}  road={r:.3f}  margin={m:+.3f}  entropy={e:.2f}')


LAYER 0   residual stream:  before + delta = after   (first 4 of 768 dims)
token      before                        delta                          |delta|
-------------------------------------------------------------------------------
The          -0.09  -0.22   0.07  -0.05     0.24  -0.58   0.33   0.39     130.8
dog           0.12  -0.13  -0.06  -0.05     0.99  -1.71   0.08   0.48      56.2
did           0.01  -0.24   0.29   0.08     0.18  -0.09   1.05   0.16      55.8
not          -0.01  -0.04   0.16   0.07     1.07   0.36   0.16  -0.13      55.1
cross         0.10  -0.06   0.18  -0.13     0.63   0.25  -2.44   1.89      55.8
the          -0.03  -0.03   0.17   0.04    -0.77   0.93  -0.71   0.77      54.5
road         -0.02  -0.01   0.14  -0.10    -0.91  -1.65  -2.11   0.42      57.3
because       0.08   0.04   0.14  -0.16    -0.00   0.76  -1.66   1.51      55.4
it            0.02  -0.05   0.14   0.05    -1.38   0.24  -0.21   0.52      54.2  <-- "it"
was          -0.07  -0.08   0.30  